# 63 Generate Final Report Figures

This notebook regenerates the final Chapter 3 and Chapter 4 report figures from saved manifests, generated-image manifests, GAN KID files, and CNN result folders. It does not retrain any GAN or CNN model. It does not depend on precomputed distribution-table exports.


In [1]:
# Configuration cell. Keep all paths and figure settings in one place.
import os
import json
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "results").exists() and (CURRENT_DIR / "notebooks").exists():
    REPO_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "results").exists() and (CURRENT_DIR.parent / "notebooks").exists():
    REPO_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError("Run this notebook from the FYP2 repository root or from the notebooks folder.")

(REPO_ROOT / ".cache").mkdir(exist_ok=True)
(REPO_ROOT / ".matplotlib_cache").mkdir(exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(REPO_ROOT / ".cache"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
from PIL import Image

FIGURE_DIR = REPO_ROOT / "figures" / "final_report_updated"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

KID_SNAPSHOTS = [40, 80, 120, 160, 200]
DATASET_ORDER = ["DIBaS", "Dataset-Bacteria"]
GAN_ORDER = ["StyleGAN2-ADA", "StyleGAN3-T", "Diffusion-GAN"]
SETTING_ORDER = ["Baseline", "StyleGAN2-ADA", "StyleGAN3-T", "Diffusion-GAN"]
MODEL_ORDER = [
    "ResNet-50",
    "EfficientNetV2-S",
    "ConvNeXtV2-Nano",
    "MobileNetV4-Conv-Medium",
    "StarNet-S4",
]

PATCH_MANIFESTS = [
    ("DIBaS", "dibas", REPO_ROOT / "manifests" / "patch_manifest_224.csv"),
    ("Dataset-Bacteria", "dataset2", REPO_ROOT / "manifests" / "dataset2_patch_manifest_224.csv"),
]

KID_FILES = [
    ("DIBaS", "StyleGAN2-ADA", REPO_ROOT / "results" / "35_select_stylegan2_sparse_snapshots_dibas" / "kid_snapshot_results_sparse.csv"),
    ("DIBaS", "StyleGAN3-T", REPO_ROOT / "results" / "24_train_stylegan3_make_images_dibas" / "kid_snapshot_results.csv"),
    ("DIBaS", "Diffusion-GAN", REPO_ROOT / "results" / "39_train_diffusion_gan_make_images_dibas" / "kid_snapshot_results.csv"),
    ("Dataset-Bacteria", "StyleGAN2-ADA", REPO_ROOT / "results" / "36_select_stylegan2_sparse_snapshots_dataset2" / "kid_snapshot_results_sparse.csv"),
    ("Dataset-Bacteria", "StyleGAN3-T", REPO_ROOT / "results" / "25_train_stylegan3_make_images_dataset2" / "kid_snapshot_results.csv"),
    ("Dataset-Bacteria", "Diffusion-GAN", REPO_ROOT / "results" / "40_train_diffusion_gan_make_images_dataset2" / "kid_snapshot_results.csv"),
]

GENERATED_MANIFESTS = [
    ("DIBaS", "StyleGAN2-ADA", REPO_ROOT / "manifests" / "stylegan2_sparse_generated_manifest_224.csv"),
    ("DIBaS", "StyleGAN3-T", REPO_ROOT / "manifests" / "stylegan3_generated_manifest_224.csv"),
    ("DIBaS", "Diffusion-GAN", REPO_ROOT / "manifests" / "diffusion_gan_generated_manifest_224.csv"),
    ("Dataset-Bacteria", "StyleGAN2-ADA", REPO_ROOT / "manifests" / "dataset2_stylegan2_sparse_generated_manifest_224.csv"),
    ("Dataset-Bacteria", "StyleGAN3-T", REPO_ROOT / "manifests" / "dataset2_stylegan3_generated_manifest_224.csv"),
    ("Dataset-Bacteria", "Diffusion-GAN", REPO_ROOT / "manifests" / "dataset2_diffusion_gan_generated_manifest_224.csv"),
]

# These are the final formal CNN result folders used in the report.
# The list is explicit to avoid accidentally reading pilot, abandoned, or old result folders.
FINAL_CNN_RESULTS = [
    ("DIBaS", "ResNet-50", "Baseline", "05_resnet50_baseline_dibas"),
    ("DIBaS", "ResNet-50", "StyleGAN2-ADA", "06_resnet50_gan_dibas"),
    ("DIBaS", "ResNet-50", "StyleGAN3-T", "26_resnet50_stylegan3_dibas"),
    ("DIBaS", "ResNet-50", "Diffusion-GAN", "41_resnet50_diffusion_gan_dibas"),
    ("DIBaS", "EfficientNetV2-S", "Baseline", "07_efficientnetv2s_baseline_dibas"),
    ("DIBaS", "EfficientNetV2-S", "StyleGAN2-ADA", "08_efficientnetv2s_gan_dibas"),
    ("DIBaS", "EfficientNetV2-S", "StyleGAN3-T", "27_efficientnetv2s_stylegan3_dibas"),
    ("DIBaS", "EfficientNetV2-S", "Diffusion-GAN", "42_efficientnetv2s_diffusion_gan_dibas"),
    ("DIBaS", "ConvNeXtV2-Nano", "Baseline", "09_convnextv2nano_baseline_dibas"),
    ("DIBaS", "ConvNeXtV2-Nano", "StyleGAN2-ADA", "10_convnextv2nano_gan_dibas"),
    ("DIBaS", "ConvNeXtV2-Nano", "StyleGAN3-T", "28_convnextv2nano_stylegan3_dibas"),
    ("DIBaS", "ConvNeXtV2-Nano", "Diffusion-GAN", "43_convnextv2nano_diffusion_gan_dibas"),
    ("DIBaS", "MobileNetV4-Conv-Medium", "Baseline", "47_mobilenetv4_baseline_dibas"),
    ("DIBaS", "MobileNetV4-Conv-Medium", "StyleGAN2-ADA", "48_mobilenetv4_gan_dibas"),
    ("DIBaS", "MobileNetV4-Conv-Medium", "StyleGAN3-T", "49_mobilenetv4_stylegan3_dibas"),
    ("DIBaS", "MobileNetV4-Conv-Medium", "Diffusion-GAN", "50_mobilenetv4_diffusion_gan_dibas"),
    ("DIBaS", "StarNet-S4", "Baseline", "51_starnet_baseline_dibas"),
    ("DIBaS", "StarNet-S4", "StyleGAN2-ADA", "52_starnet_gan_dibas"),
    ("DIBaS", "StarNet-S4", "StyleGAN3-T", "53_starnet_stylegan3_dibas"),
    ("DIBaS", "StarNet-S4", "Diffusion-GAN", "54_starnet_diffusion_gan_dibas"),
    ("Dataset-Bacteria", "ResNet-50", "Baseline", "15_resnet50_baseline_dataset2"),
    ("Dataset-Bacteria", "ResNet-50", "StyleGAN2-ADA", "16_resnet50_gan_dataset2"),
    ("Dataset-Bacteria", "ResNet-50", "StyleGAN3-T", "29_resnet50_stylegan3_dataset2"),
    ("Dataset-Bacteria", "ResNet-50", "Diffusion-GAN", "44_resnet50_diffusion_gan_dataset2"),
    ("Dataset-Bacteria", "EfficientNetV2-S", "Baseline", "17_efficientnetv2s_baseline_dataset2"),
    ("Dataset-Bacteria", "EfficientNetV2-S", "StyleGAN2-ADA", "18_efficientnetv2s_gan_dataset2"),
    ("Dataset-Bacteria", "EfficientNetV2-S", "StyleGAN3-T", "30_efficientnetv2s_stylegan3_dataset2"),
    ("Dataset-Bacteria", "EfficientNetV2-S", "Diffusion-GAN", "45_efficientnetv2s_diffusion_gan_dataset2"),
    ("Dataset-Bacteria", "ConvNeXtV2-Nano", "Baseline", "19_convnextv2nano_baseline_dataset2"),
    ("Dataset-Bacteria", "ConvNeXtV2-Nano", "StyleGAN2-ADA", "20_convnextv2nano_gan_dataset2"),
    ("Dataset-Bacteria", "ConvNeXtV2-Nano", "StyleGAN3-T", "31_convnextv2nano_stylegan3_dataset2"),
    ("Dataset-Bacteria", "ConvNeXtV2-Nano", "Diffusion-GAN", "46_convnextv2nano_diffusion_gan_dataset2"),
    ("Dataset-Bacteria", "MobileNetV4-Conv-Medium", "Baseline", "55_mobilenetv4_baseline_dataset2"),
    ("Dataset-Bacteria", "MobileNetV4-Conv-Medium", "StyleGAN2-ADA", "56_mobilenetv4_gan_dataset2"),
    ("Dataset-Bacteria", "MobileNetV4-Conv-Medium", "StyleGAN3-T", "57_mobilenetv4_stylegan3_dataset2"),
    ("Dataset-Bacteria", "MobileNetV4-Conv-Medium", "Diffusion-GAN", "58_mobilenetv4_diffusion_gan_dataset2"),
    ("Dataset-Bacteria", "StarNet-S4", "Baseline", "59_starnet_baseline_dataset2"),
    ("Dataset-Bacteria", "StarNet-S4", "StyleGAN2-ADA", "60_starnet_gan_dataset2"),
    ("Dataset-Bacteria", "StarNet-S4", "StyleGAN3-T", "61_starnet_stylegan3_dataset2"),
    ("Dataset-Bacteria", "StarNet-S4", "Diffusion-GAN", "62_starnet_diffusion_gan_dataset2"),
]

EXPECTED_TRAINING_MODE = {
    "Baseline": "baseline",
    "StyleGAN2-ADA": "gan",
    "StyleGAN3-T": "stylegan3",
    "Diffusion-GAN": "diffusion_gan",
}

GAN_COLOURS = {
    "StyleGAN2-ADA": "#0072B2",
    "StyleGAN3-T": "#009E73",
    "Diffusion-GAN": "#D55E00",
}
SETTING_STYLES = {
    "Baseline": ("#4D4D4D", "o"),
    "StyleGAN2-ADA": ("#0072B2", "s"),
    "StyleGAN3-T": ("#009E73", "^"),
    "Diffusion-GAN": ("#D55E00", "D"),
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 120,
})

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Required file is missing: {path}")

def read_json(path: Path) -> dict:
    require_file(path)
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_figure(fig, stem: str) -> tuple[Path, Path]:
    png_path = FIGURE_DIR / f"{stem}.png"
    pdf_path = FIGURE_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    return png_path, pdf_path

print("Repository root:", REPO_ROOT)
print("Figure output folder:", FIGURE_DIR)


Repository root: C:\Users\FYP2610\Downloads\FYP2
Figure output folder: C:\Users\FYP2610\Downloads\FYP2\figures\final_report_updated


In [2]:
# Load and validate all source files before drawing any figure.
kid_frames = []
for dataset_name, gan_name, csv_path in KID_FILES:
    require_file(csv_path)
    current = pd.read_csv(csv_path)
    required_columns = {"fold_id", "snapshot_kimg", "kid50k_full", "snapshot_path"}
    if not required_columns.issubset(current.columns):
        raise ValueError(f"KID file has missing columns: {csv_path}")
    if sorted(current["snapshot_kimg"].unique().tolist()) != KID_SNAPSHOTS:
        raise ValueError(f"Unexpected snapshot kimg values in {csv_path}")
    if sorted(current["fold_id"].unique().tolist()) != [1, 2, 3, 4, 5]:
        raise ValueError(f"Unexpected fold IDs in {csv_path}")
    current = current.copy()
    current["dataset"] = dataset_name
    current["gan"] = gan_name
    kid_frames.append(current)

kid_df = pd.concat(kid_frames, ignore_index=True)
if len(kid_df) != 150:
    raise ValueError(f"Expected 150 KID rows, but found {len(kid_df)}")

manifest_frames = []
for dataset_name, gan_name, manifest_path in GENERATED_MANIFESTS:
    require_file(manifest_path)
    current = pd.read_csv(manifest_path)
    required_columns = {"fold_id", "seed", "file_path", "binary_group", "image_size"}
    if not required_columns.issubset(current.columns):
        raise ValueError(f"Generated manifest has missing columns: {manifest_path}")
    if set(current["binary_group"].unique()) != {"cocci"}:
        raise ValueError(f"Generated manifest is not cocci-only: {manifest_path}")
    if set(current["image_size"].unique()) != {224}:
        raise ValueError(f"Generated manifest is not 224 x 224: {manifest_path}")
    current = current.copy()
    current["dataset"] = dataset_name
    current["gan"] = gan_name
    manifest_frames.append(current)

generated_df = pd.concat(manifest_frames, ignore_index=True)

# Rebuild the fold-specific balancing table directly from the real 224 x 224 patch manifests.
balance_rows = []
for dataset_display_name, dataset_key, manifest_path in PATCH_MANIFESTS:
    require_file(manifest_path)
    patch_df = pd.read_csv(manifest_path)
    required_columns = {"fold_id", "binary_group", "binary_label", "patch_size", "source_type"}
    if not required_columns.issubset(patch_df.columns):
        raise ValueError(f"Patch manifest has missing columns: {manifest_path}")
    if set(patch_df["patch_size"].unique()) != {224}:
        raise ValueError(f"Patch manifest is not 224 x 224: {manifest_path}")
    if set(patch_df["source_type"].unique()) != {"real"}:
        raise ValueError(f"Patch manifest contains non-real patches: {manifest_path}")

    grouped = patch_df.groupby(["fold_id", "binary_group"]).size().unstack(fill_value=0)
    for fold_id in [1, 2, 3, 4, 5]:
        if fold_id not in grouped.index:
            raise ValueError(f"Missing fold {fold_id} in {manifest_path}")
        cocci_count = int(grouped.loc[fold_id].get("cocci", 0))
        bacilli_count = int(grouped.loc[fold_id].get("bacilli", 0))
        needed_gan_count = bacilli_count - cocci_count
        if needed_gan_count < 0:
            raise ValueError(f"Cocci is not the minority class for {dataset_display_name}, fold {fold_id}")
        balance_rows.append({
            "dataset_name": dataset_key,
            "dataset": dataset_display_name,
            "fold_id": fold_id,
            "cocci": cocci_count,
            "bacilli": bacilli_count,
            "needed_gan_count_to_balance_train_fold": needed_gan_count,
            "final_cocci": cocci_count + needed_gan_count,
        })

balance_plan_df = pd.DataFrame(balance_rows)
if len(balance_plan_df) != 10:
    raise ValueError(f"Expected 10 balance rows, but found {len(balance_plan_df)}")
if not (balance_plan_df["final_cocci"] == balance_plan_df["bacilli"]).all():
    raise ValueError("GAN top-up does not exactly balance cocci to bacilli in the rebuilt balance plan.")

# Rebuild the CNN summary directly from each formal result folder.
summary_rows = []
for model_order, (dataset_name, model_display_name, setting, result_dir_name) in enumerate(FINAL_CNN_RESULTS):
    result_dir = REPO_ROOT / "results" / result_dir_name
    summary_path = result_dir / "summary.json"
    overall_path = result_dir / "overall_metrics_mean.csv"
    summary = read_json(summary_path)
    require_file(overall_path)
    overall_df = pd.read_csv(overall_path)
    if len(overall_df) != 1:
        raise ValueError(f"Expected one overall metric row in {overall_path}, but found {len(overall_df)}")
    overall = overall_df.iloc[0]

    if summary.get("model_display_name") != model_display_name:
        raise ValueError(f"Model name mismatch in {summary_path}: expected {model_display_name}, found {summary.get('model_display_name')}")
    if summary.get("training_mode") != EXPECTED_TRAINING_MODE[setting]:
        raise ValueError(f"Training mode mismatch in {summary_path}: expected {EXPECTED_TRAINING_MODE[setting]}, found {summary.get('training_mode')}")

    summary_rows.append({
        "result_dir": result_dir_name,
        "model_name": summary.get("model_name"),
        "model_display_name": model_display_name,
        "training_mode": summary.get("training_mode"),
        "epochs": int(summary.get("epochs", overall.get("epochs", 0))),
        "seed_count": int(summary.get("seed_count", overall["seed_count"])),
        "fold_count": int(summary.get("fold_count", overall["fold_count"])),
        "run_count": int(summary.get("run_count", overall["run_count"])),
        "patch_manifest_path": summary.get("patch_manifest_path", ""),
        "gan_manifest_path": summary.get("gan_manifest_path", ""),
        "dataset": dataset_name,
        "accuracy_mean": float(overall["mean_accuracy"]),
        "accuracy_std": float(overall["std_accuracy"]),
        "balanced_accuracy_mean": float(overall["mean_balanced_accuracy"]),
        "balanced_accuracy_std": float(overall["std_balanced_accuracy"]),
        "macro_precision_mean": float(overall["mean_macro_precision"]),
        "macro_precision_std": float(overall["std_macro_precision"]),
        "macro_recall_mean": float(overall["mean_macro_recall"]),
        "macro_recall_std": float(overall["std_macro_recall"]),
        "macro_f1_mean": float(overall["mean_macro_f1"]),
        "macro_f1_std": float(overall["std_macro_f1"]),
        "setting": setting,
        "model_order": MODEL_ORDER.index(model_display_name),
        "mode_order": SETTING_ORDER.index(setting),
    })

cnn_summary_df = pd.DataFrame(summary_rows)
expected_summary_rows = len(DATASET_ORDER) * len(MODEL_ORDER) * len(SETTING_ORDER)
if len(cnn_summary_df) != expected_summary_rows:
    raise ValueError(f"Expected {expected_summary_rows} CNN summary rows, but found {len(cnn_summary_df)}")
if sorted(cnn_summary_df["dataset"].unique().tolist()) != sorted(DATASET_ORDER):
    raise ValueError("CNN summary contains unexpected dataset names.")
if sorted(cnn_summary_df["setting"].unique().tolist()) != sorted(SETTING_ORDER):
    raise ValueError("CNN summary contains unexpected training settings.")
if sorted(cnn_summary_df["model_display_name"].unique().tolist()) != sorted(MODEL_ORDER):
    raise ValueError("CNN summary contains unexpected CNN models.")

print("KID rows:", len(kid_df))
print("Generated manifest rows:", len(generated_df))
print("Rebuilt balance plan rows:", len(balance_plan_df))
print("Rebuilt CNN summary rows:", len(cnn_summary_df))


KID rows: 150
Generated manifest rows: 5865
Rebuilt balance plan rows: 10
Rebuilt CNN summary rows: 40


In [3]:
# Figure 3.4 shows the exact fold-specific rule used to add generated cocci patches.
fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(11.0, 4.2), sharey=False)
bar_width = 0.28

for ax, dataset_name in zip(axes, DATASET_ORDER):
    current = balance_plan_df[balance_plan_df["dataset"] == dataset_name].sort_values("fold_id")
    if len(current) != 5:
        raise ValueError(f"Expected five balance rows for {dataset_name}, but found {len(current)}")

    x = np.arange(len(current))
    real_cocci = current["cocci"].to_numpy()
    gan_added = current["needed_gan_count_to_balance_train_fold"].to_numpy()
    real_bacilli = current["bacilli"].to_numpy()

    ax.bar(x - bar_width / 2, real_cocci, width=bar_width, color="#56B4E9", label="Real cocci")
    ax.bar(x - bar_width / 2, gan_added, width=bar_width, bottom=real_cocci, color="#E69F00", label="Generated cocci added")
    ax.bar(x + bar_width / 2, real_bacilli, width=bar_width, color="#009E73", label="Real bacilli")

    for position, final_cocci, bacilli in zip(x, real_cocci + gan_added, real_bacilli):
        ax.plot([position - bar_width / 2, position + bar_width / 2], [final_cocci, bacilli], color="#333333", linewidth=1.0)

    ax.set_title(dataset_name)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Fold {fold_id}" for fold_id in current["fold_id"]])
    ax.set_ylabel("Training patches")
    ax.grid(axis="y", alpha=0.20, linewidth=0.7)
    ax.set_axisbelow(True)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.03))
fig.suptitle("Fold-specific GAN minority balancing rule", y=1.12, fontsize=13, fontweight="bold")
fig.text(0.5, -0.02, "Generated cocci patches were added only to the training fold until cocci matched the real bacilli count.", ha="center", fontsize=10)
fig.tight_layout(rect=(0, 0.04, 1, 0.94))

fig_3_4_png, fig_3_4_pdf = save_figure(fig, "report_figure_3_4_fold_specific_gan_minority_balancing_counts")
print(fig_3_4_png.relative_to(REPO_ROOT))

figures\final_report_updated\report_figure_3_4_fold_specific_gan_minority_balancing_counts.png


In [4]:
# Figure 4.1 shows the KID rule used to select fold-specific GAN snapshots.
fig, axes = plt.subplots(len(DATASET_ORDER), len(GAN_ORDER), figsize=(12.0, 6.4), sharex=True)

for row_index, dataset_name in enumerate(DATASET_ORDER):
    for col_index, gan_name in enumerate(GAN_ORDER):
        ax = axes[row_index, col_index]
        current = kid_df[(kid_df["dataset"] == dataset_name) & (kid_df["gan"] == gan_name)]
        colour = GAN_COLOURS[gan_name]
        for fold_id, fold_df in current.groupby("fold_id"):
            fold_df = fold_df.sort_values("snapshot_kimg")
            ax.plot(fold_df["snapshot_kimg"], fold_df["kid50k_full"], color=colour, alpha=0.25, linewidth=1.0)
            best_row = fold_df.loc[fold_df["kid50k_full"].idxmin()]
            ax.scatter(best_row["snapshot_kimg"], best_row["kid50k_full"], s=30, facecolors="white", edgecolors="black", linewidths=1.0, zorder=5)
        mean_df = current.groupby("snapshot_kimg", as_index=False)["kid50k_full"].mean().sort_values("snapshot_kimg")
        ax.plot(mean_df["snapshot_kimg"], mean_df["kid50k_full"], color=colour, marker="o", linewidth=2.0, markersize=4)
        ax.set_title(f"{dataset_name}\n{gan_name}")
        ax.set_xticks(KID_SNAPSHOTS)
        ax.grid(alpha=0.20, linewidth=0.7)
        if col_index == 0:
            ax.set_ylabel("KID50k full")
        if row_index == len(DATASET_ORDER) - 1:
            ax.set_xlabel("Snapshot Kimg")

legend_handles = [
    Line2D([0], [0], color="black", marker="o", markerfacecolor="white", linewidth=0, label="Selected lowest-KID fold snapshot"),
    Line2D([0], [0], color="black", marker="o", linewidth=2, label="Mean KID across folds"),
]
fig.legend(handles=legend_handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.03))
fig.suptitle("KID Progress for Fold-Specific GAN Snapshot Selection", y=1.09, fontsize=13, fontweight="bold")
fig.tight_layout()
fig_4_1_png, fig_4_1_pdf = save_figure(fig, "report_figure_4_1_kid_snapshot_selection_updated")
print(fig_4_1_png.relative_to(REPO_ROOT))

figures\final_report_updated\report_figure_4_1_kid_snapshot_selection_updated.png


In [5]:
# Figure 4.2 shows deterministic examples from the generated cocci manifests.
RESAMPLE = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS

def make_montage(relative_paths, tile_size=112, gap=6):
    if len(relative_paths) != 4:
        raise ValueError("Each montage must contain exactly four image paths.")
    canvas = Image.new("RGB", (tile_size * 2 + gap, tile_size * 2 + gap), "white")
    for index, relative_path in enumerate(relative_paths):
        image_path = REPO_ROOT / relative_path
        require_file(image_path)
        image = Image.open(image_path).convert("RGB").resize((tile_size, tile_size), RESAMPLE)
        x_position = (index % 2) * (tile_size + gap)
        y_position = (index // 2) * (tile_size + gap)
        canvas.paste(image, (x_position, y_position))
    return np.asarray(canvas)

fig, axes = plt.subplots(len(DATASET_ORDER), len(GAN_ORDER), figsize=(9.6, 6.4))

for row_index, dataset_name in enumerate(DATASET_ORDER):
    for col_index, gan_name in enumerate(GAN_ORDER):
        ax = axes[row_index, col_index]
        current = generated_df[(generated_df["dataset"] == dataset_name) & (generated_df["gan"] == gan_name)]
        preview_rows = current[current["fold_id"] == 1].sort_values("seed").head(4)
        if len(preview_rows) != 4:
            raise ValueError(f"Not enough preview images for {dataset_name} / {gan_name}")
        montage = make_montage(preview_rows["file_path"].tolist())
        ax.imshow(montage)
        ax.set_title(f"{dataset_name}\n{gan_name}")
        ax.axis("off")

fig.suptitle("Generated Cocci Patch Examples from Fold 1", y=0.99, fontsize=13, fontweight="bold")
fig.tight_layout()
fig_4_2_png, fig_4_2_pdf = save_figure(fig, "report_figure_4_2_generated_cocci_examples_updated")
print(fig_4_2_png.relative_to(REPO_ROOT))

figures\final_report_updated\report_figure_4_2_generated_cocci_examples_updated.png


In [6]:
# Figure 4.3 compares patch-level macro F1-score gains over each matched baseline.
def draw_metric_gain_heatmap(metric_mean, colourbar_label, title, stem):
    gan_settings = SETTING_ORDER[1:]
    gain_rows = []
    for dataset_name in DATASET_ORDER:
        dataset_df = cnn_plot_df[cnn_plot_df["dataset"] == dataset_name]
        for model_name in MODEL_ORDER:
            model_df = dataset_df[dataset_df["model_display_name"] == model_name].set_index("setting")
            if "Baseline" not in model_df.index:
                raise ValueError(f"Missing baseline row for {dataset_name} / {model_name}")
            baseline_value = float(model_df.loc["Baseline", metric_mean])
            for setting in gan_settings:
                if setting not in model_df.index:
                    raise ValueError(f"Missing {setting} row for {dataset_name} / {model_name}")
                gain_rows.append({
                    "dataset": dataset_name,
                    "model_display_name": model_name,
                    "setting": setting,
                    "gain_pp": (float(model_df.loc[setting, metric_mean]) - baseline_value) * 100.0,
                })

    gain_df = pd.DataFrame(gain_rows)
    max_abs_gain = float(np.nanmax(np.abs(gain_df["gain_pp"])))
    colour_limit = max(1.0, np.ceil(max_abs_gain))
    norm = TwoSlopeNorm(vmin=-colour_limit, vcenter=0.0, vmax=colour_limit)
    x_labels = ["StyleGAN2-ADA", "StyleGAN3-T", "Diffusion-GAN"]

    fig = plt.figure(figsize=(12.8, 5.3))
    grid = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.0, 0.045], wspace=0.12)
    first_axis = fig.add_subplot(grid[0, 0])
    second_axis = fig.add_subplot(grid[0, 1], sharey=first_axis)
    colourbar_axis = fig.add_subplot(grid[0, 2])
    axes = [first_axis, second_axis]
    last_image = None
    for ax, dataset_name in zip(axes, DATASET_ORDER):
        current = gain_df[gain_df["dataset"] == dataset_name]
        heatmap = current.pivot(index="model_display_name", columns="setting", values="gain_pp").loc[MODEL_ORDER, gan_settings]
        last_image = ax.imshow(heatmap.to_numpy(), cmap="RdBu", norm=norm, aspect="auto")

        ax.set_title(dataset_name)
        ax.set_xticks(np.arange(len(gan_settings)))
        ax.set_xticklabels(x_labels)
        ax.set_yticks(np.arange(len(MODEL_ORDER)))
        ax.set_yticklabels(MODEL_ORDER)
        if ax is not first_axis:
            ax.tick_params(labelleft=False)
        ax.tick_params(axis="x", length=0)
        ax.tick_params(axis="y", length=0)
        ax.set_xticks(np.arange(-0.5, len(gan_settings), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(MODEL_ORDER), 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=1.5)
        ax.tick_params(which="minor", bottom=False, left=False)

        for y_index, model_name in enumerate(MODEL_ORDER):
            for x_index, setting in enumerate(gan_settings):
                value = heatmap.loc[model_name, setting]
                text_colour = "white" if abs(value) > colour_limit * 0.58 else "black"
                ax.text(x_index, y_index, f"{value:+.2f}", ha="center", va="center", fontsize=8.5, color=text_colour, fontweight="bold")

    axes[0].set_ylabel("CNN Backbone")
    cbar = fig.colorbar(last_image, cax=colourbar_axis)
    cbar.set_label(colourbar_label)
    fig.suptitle(title, y=0.98, fontsize=13, fontweight="bold")
    fig.text(0.5, 0.045, "Cell values show the mean gain over the matched baseline in percentage points.", ha="center", fontsize=9)
    fig.subplots_adjust(left=0.18, right=0.95, bottom=0.23, top=0.82)
    return save_figure(fig, stem)

cnn_plot_df = cnn_summary_df.copy()
fig_4_3_png, fig_4_3_pdf = draw_metric_gain_heatmap(
    metric_mean="macro_f1_mean",
    colourbar_label="Gain Over Baseline (Percentage Points)",
    title="Patch-Level Macro F1-Score Gain Over Baseline",
    stem="report_figure_4_3_macro_f1_gain_heatmap_updated",
)
print(fig_4_3_png.relative_to(REPO_ROOT))

figures\final_report_updated\report_figure_4_3_macro_f1_gain_heatmap_updated.png


In [7]:
# Figure 4.4 focuses on cocci recall because cocci is the minority class that GAN balancing targets.
cocci_rows = []
for _, row in cnn_summary_df.iterrows():
    class_metric_path = REPO_ROOT / "results" / row["result_dir"] / "class_metrics_mean.csv"
    require_file(class_metric_path)
    class_df = pd.read_csv(class_metric_path)
    if "cocci" not in set(class_df["class_name"]):
        raise ValueError(f"Cocci class is missing from {class_metric_path}")
    cocci_row = class_df[class_df["class_name"] == "cocci"].iloc[0]
    cocci_rows.append({
        "dataset": row["dataset"],
        "model_display_name": row["model_display_name"],
        "setting": row["setting"],
        "cocci_recall_mean": cocci_row["mean_recall"],
        "cocci_recall_std": cocci_row["std_recall"],
    })

cnn_plot_df = pd.DataFrame(cocci_rows)
if len(cnn_plot_df) != expected_summary_rows:
    raise ValueError(f"Expected {expected_summary_rows} cocci recall rows, but found {len(cnn_plot_df)}")

fig_4_4_png, fig_4_4_pdf = draw_metric_gain_heatmap(
    metric_mean="cocci_recall_mean",
    colourbar_label="Gain Over Baseline (Percentage Points)",
    title="Minority Cocci Recall Gain Over Baseline",
    stem="report_figure_4_4_minority_cocci_recall_gain_heatmap_updated",
)
print(fig_4_4_png.relative_to(REPO_ROOT))

figures\final_report_updated\report_figure_4_4_minority_cocci_recall_gain_heatmap_updated.png


In [8]:
# Save a small manifest so the report figures can be audited later.
chapter4_manifest = pd.DataFrame([
    {"figure": "Figure 4.1", "title": "KID Progress for GAN Snapshot Selection", "file_path": str(fig_4_1_png.relative_to(REPO_ROOT)), "pdf_path": str(fig_4_1_pdf.relative_to(REPO_ROOT))},
    {"figure": "Figure 4.2", "title": "Generated Cocci Patch Examples", "file_path": str(fig_4_2_png.relative_to(REPO_ROOT)), "pdf_path": str(fig_4_2_pdf.relative_to(REPO_ROOT))},
    {"figure": "Figure 4.3", "title": "Patch-Level Macro F1-Score Gain Over Baseline", "file_path": str(fig_4_3_png.relative_to(REPO_ROOT)), "pdf_path": str(fig_4_3_pdf.relative_to(REPO_ROOT))},
    {"figure": "Figure 4.4", "title": "Minority Cocci Recall Gain Over Baseline", "file_path": str(fig_4_4_png.relative_to(REPO_ROOT)), "pdf_path": str(fig_4_4_pdf.relative_to(REPO_ROOT))},
])
chapter4_manifest_path = FIGURE_DIR / "updated_chapter4_figure_manifest.csv"
chapter4_manifest.to_csv(chapter4_manifest_path, index=False)

# Figure 3.1, 3.2, 3.3, and 3.5 are manually prepared report figures.
# They are included in the full manifest only when the files are present on this machine.
static_manifest_rows = [
    ("Figure 3.1", "Overall Study Workflow", "figures/report_figure_3_1_overall_study_workflow.png"),
    ("Figure 3.2", "Example Real Microscopy Patches from the Two Datasets", "figures/report_figure_3_2_real_dataset_patch_examples.png"),
    ("Figure 3.3", "Source-Level Split Before Patch Extraction", "figures/report_figure_3_3_source_level_split.png"),
    ("Figure 3.5", "Research Project Gantt Chart", "figures/report_figure_3_5_research_project_gantt_chart.png"),
]
generated_manifest_rows = [
    ("Figure 3.4", "Fold-specific GAN Minority Balancing Rule", str(fig_3_4_png.relative_to(REPO_ROOT))),
]
for _, row in chapter4_manifest.iterrows():
    generated_manifest_rows.append((row["figure"], row["title"], row["file_path"]))

generated_manifest = pd.DataFrame(generated_manifest_rows, columns=["figure", "title", "file_path"])
generated_manifest["exists"] = generated_manifest["file_path"].apply(lambda p: (REPO_ROOT / p).exists())
if not generated_manifest["exists"].all():
    missing = generated_manifest[~generated_manifest["exists"]]
    raise FileNotFoundError("Some generated report figures are missing:\n" + missing.to_string(index=False))

static_manifest = pd.DataFrame(static_manifest_rows, columns=["figure", "title", "file_path"])
static_manifest["exists"] = static_manifest["file_path"].apply(lambda p: (REPO_ROOT / p).exists())

final_manifest = pd.concat([static_manifest, generated_manifest], ignore_index=True)
final_manifest_path = REPO_ROOT / "figures" / "final_report_figure_manifest.csv"
if static_manifest["exists"].all():
    final_manifest.to_csv(final_manifest_path, index=False)
    print("Saved:", final_manifest_path.relative_to(REPO_ROOT))
else:
    missing_static = static_manifest[~static_manifest["exists"]]
    print("Static report figures are not present on this machine, so the full final figure manifest was not overwritten.")
    print(missing_static[["figure", "file_path"]].to_string(index=False))

display(chapter4_manifest)
display(final_manifest)
print("Saved:", chapter4_manifest_path.relative_to(REPO_ROOT))


Static report figures are not present on this machine, so the full final figure manifest was not overwritten.
    figure                                                  file_path
Figure 3.1       figures/report_figure_3_1_overall_study_workflow.png
Figure 3.2  figures/report_figure_3_2_real_dataset_patch_examples.png
Figure 3.3           figures/report_figure_3_3_source_level_split.png
Figure 3.5 figures/report_figure_3_5_research_project_gantt_chart.png


,figure,title,file_path,pdf_path
0,Figure 4.1,KID Progress for GAN Snapshot Selection,figures\final_report_updated\report_figure_4_1...,figures\final_report_updated\report_figure_4_1...
1,Figure 4.2,Generated Cocci Patch Examples,figures\final_report_updated\report_figure_4_2...,figures\final_report_updated\report_figure_4_2...
2,Figure 4.3,Patch-Level Macro F1-Score Gain Over Baseline,figures\final_report_updated\report_figure_4_3...,figures\final_report_updated\report_figure_4_3...
3,Figure 4.4,Minority Cocci Recall Gain Over Baseline,figures\final_report_updated\report_figure_4_4...,figures\final_report_updated\report_figure_4_4...


,figure,title,file_path,exists
0,Figure 3.1,Overall Study Workflow,figures/report_figure_3_1_overall_study_workfl...,False
1,Figure 3.2,Example Real Microscopy Patches from the Two D...,figures/report_figure_3_2_real_dataset_patch_e...,False
2,Figure 3.3,Source-Level Split Before Patch Extraction,figures/report_figure_3_3_source_level_split.png,False
3,Figure 3.5,Research Project Gantt Chart,figures/report_figure_3_5_research_project_gan...,False
4,Figure 3.4,Fold-specific GAN Minority Balancing Rule,figures\final_report_updated\report_figure_3_4...,True
5,Figure 4.1,KID Progress for GAN Snapshot Selection,figures\final_report_updated\report_figure_4_1...,True
6,Figure 4.2,Generated Cocci Patch Examples,figures\final_report_updated\report_figure_4_2...,True
7,Figure 4.3,Patch-Level Macro F1-Score Gain Over Baseline,figures\final_report_updated\report_figure_4_3...,True
8,Figure 4.4,Minority Cocci Recall Gain Over Baseline,figures\final_report_updated\report_figure_4_4...,True


Saved: figures\final_report_updated\updated_chapter4_figure_manifest.csv
